# Retrieval Methods on SciFact: A Controlled Comparison

**Question.** How do classical and neural retrieval methods compare when they are evaluated against human relevance judgements, not just inspected by eye?

The [toy walkthrough](01_methods_walkthrough.ipynb) shows *how* seven methods rank eight sentences. This notebook measures *how well* they rank, on a standard benchmark with known correct answers.

| Group | Method | Trained on |
|---|---|---|
| Exact match | Boolean retrieval (AND, OR) | nothing |
| Lexical | TF-IDF, **BM25** (standard baseline) | corpus statistics |
| Static embeddings, trained here | Word2Vec CBOW, Word2Vec Skip-gram, FastText | the SciFact abstracts |
| Static embeddings, pretrained | GloVe (Wikipedia + Gigaword, 100-d) | external text |
| Contextual, pretrained | Sentence-BERT `all-MiniLM-L6-v2`, DistilBERT mean pooling | external text |

**Protocol.** One tokenizer for all lexical and static-embedding methods; all hyperparameters fixed before looking at test results (library or literature defaults); no fine-tuning; every method ranks the full corpus for every test query; metrics computed with one implementation and cross-checked against `pytrec_eval`.

**Runtime.** About 10–15 minutes on Google Colab with a GPU runtime (*Runtime → Change runtime type → T4 GPU*); longer on CPU.

## 1. Setup

In [ ]:
# Colab already has numpy, pandas, scikit-learn, scipy, torch and transformers.
!pip install -q gensim sentence-transformers pytrec-eval-terrier

In [ ]:
import os, re, json, time, random, zipfile, urllib.request
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

TOP_K = 100                       # documents kept per query (enough for Recall@100)
DATA_DIR = os.environ.get("IR_DATA_DIR", "scifact")
RUN_PRETRAINED = os.environ.get("IR_SKIP_PRETRAINED") is None   # GloVe, SBERT, DistilBERT

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"
print("Device:", DEVICE)

## 2. Data: SciFact (BEIR version)

SciFact [1] pairs scientific claims with the paper abstracts that support or refute them. In the BEIR benchmark [2] each claim is a **query**, the abstracts are the **corpus**, and an abstract is **relevant** if experts marked it as evidence for the claim. Relevance is binary (1 = relevant).

We use the official **test split** (300 queries). A document is its title followed by its abstract, as in BEIR.

In [ ]:
BEIR_URL = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip"

def download_scifact(target):
    # Official BEIR zip; if that host is unreachable, fall back to the MTEB copy on Hugging Face.
    os.makedirs(target, exist_ok=True)
    try:
        zpath, _ = urllib.request.urlretrieve(BEIR_URL, "scifact.zip")
        with zipfile.ZipFile(zpath) as z:
            for name in z.namelist():
                rel = name.split("/", 1)[1] if "/" in name else ""
                if rel and not name.endswith("/"):
                    os.makedirs(os.path.join(target, os.path.dirname(rel)), exist_ok=True)
                    with open(os.path.join(target, rel), "wb") as f:
                        f.write(z.read(name))
        print("Downloaded from BEIR.")
    except Exception as e:
        print("BEIR download failed (", e, "); using Hugging Face mteb/scifact.")
        from datasets import load_dataset
        corpus = load_dataset("mteb/scifact", "corpus", split="corpus")
        queries = load_dataset("mteb/scifact", "queries", split="queries")
        qrels = load_dataset("mteb/scifact", "default", split="test")
        with open(os.path.join(target, "corpus.jsonl"), "w") as f:
            for r in corpus:
                f.write(json.dumps({"_id": r["_id"], "title": r["title"], "text": r["text"]}) + "\n")
        with open(os.path.join(target, "queries.jsonl"), "w") as f:
            for r in queries:
                f.write(json.dumps({"_id": r["_id"], "text": r["text"]}) + "\n")
        os.makedirs(os.path.join(target, "qrels"), exist_ok=True)
        with open(os.path.join(target, "qrels", "test.tsv"), "w") as f:
            f.write("query-id\tcorpus-id\tscore\n")
            for r in qrels:
                f.write(f"{r['query-id']}\t{r['corpus-id']}\t{r['score']}\n")

if not os.path.exists(os.path.join(DATA_DIR, "corpus.jsonl")):
    download_scifact(DATA_DIR)

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

corpus = {str(d["_id"]): (d.get("title") or "", d.get("text") or "") for d in read_jsonl(os.path.join(DATA_DIR, "corpus.jsonl"))}
all_queries = {str(q["_id"]): q["text"] for q in read_jsonl(os.path.join(DATA_DIR, "queries.jsonl"))}

qrels = defaultdict(dict)
qrels_df = pd.read_csv(os.path.join(DATA_DIR, "qrels", "test.tsv"), sep="\t", dtype=str)
for qid, did, rel in qrels_df[["query-id", "corpus-id", "score"]].itertuples(index=False):
    if int(rel) > 0:
        qrels[str(qid)][str(did)] = int(rel)
qrels = dict(qrels)

doc_ids = list(corpus)
doc_texts = [f"{t} {x}".strip() for t, x in (corpus[d] for d in doc_ids)]
query_ids = sorted(q for q in qrels if q in all_queries)
query_texts = [all_queries[q] for q in query_ids]

n_rel = [len(qrels[q]) for q in query_ids]
print(f"Documents         : {len(doc_ids):,}")
print(f"Test queries      : {len(query_ids):,}")
print(f"Relevant per query: mean {np.mean(n_rel):.2f}, max {max(n_rel)}")
print(f"Words per document: median {int(np.median([len(t.split()) for t in doc_texts]))}")
print("\nExample query:", query_texts[0])

## 3. Shared preprocessing

Lowercase, keep alphanumeric tokens of two or more characters, and drop scikit-learn's English stop words. The same tokenizer feeds Boolean retrieval, TF-IDF, BM25, Word2Vec, FastText and GloVe, so differences between these methods come from the models, not from preprocessing. The transformer models use their own subword tokenizers.

In [ ]:
TOKEN_RE = re.compile(r"[a-z0-9]+")
STOP = frozenset(ENGLISH_STOP_WORDS)

def tokenize(text):
    return [t for t in TOKEN_RE.findall(text.lower()) if len(t) > 1 and t not in STOP]

doc_tokens = [tokenize(t) for t in doc_texts]
query_tokens = [tokenize(t) for t in query_texts]
print("Query tokens:", query_tokens[0])
print(f"Mean tokens: documents {np.mean([len(t) for t in doc_tokens]):.0f}, queries {np.mean([len(t) for t in query_tokens]):.1f}")

## 4. Evaluation

For each query, a method returns its top 100 documents. We report:

- **nDCG@10** (main metric, as in BEIR): rewards relevant documents near the top of the first ten results.
- **Recall@10** and **Recall@100**: the share of a query's relevant documents found in the top 10 and top 100.
- **MRR@10**: one over the rank of the first relevant document (0 if none in the top 10).

Ties are broken as in `trec_eval` (score, then document id, both descending). Our implementation is checked against `pytrec_eval` for every ranked method.

In [ ]:
def ranked_list(run_q):
    # trec_eval order: score descending, then docno descending
    return [d for d, _ in sorted(run_q.items(), key=lambda x: (x[1], x[0]), reverse=True)]

def per_query_metrics(run, qrels):
    rows = {}
    for qid in query_ids:
        rel = qrels[qid]
        ranking = ranked_list(run.get(qid, {}))
        gains = [rel.get(d, 0) for d in ranking]
        dcg = sum(g / np.log2(i + 2) for i, g in enumerate(gains[:10]))
        ideal = sorted(rel.values(), reverse=True)[:10]
        idcg = sum(g / np.log2(i + 2) for i, g in enumerate(ideal))
        first = next((i for i, g in enumerate(gains[:10]) if g > 0), None)
        rows[qid] = {
            "nDCG@10": dcg / idcg,
            "Recall@10": sum(g > 0 for g in gains[:10]) / len(rel),
            "Recall@100": sum(g > 0 for g in gains[:100]) / len(rel),
            "MRR@10": 0.0 if first is None else 1.0 / (first + 1),
        }
    return pd.DataFrame.from_dict(rows, orient="index")

def check_with_pytrec(run, per_q):
    try:
        import pytrec_eval
    except ImportError:
        return "pytrec_eval not installed"
    ev = pytrec_eval.RelevanceEvaluator({q: qrels[q] for q in query_ids}, {"ndcg_cut.10", "recall.10", "recall.100"})
    res = ev.evaluate({q: {d: float(s) for d, s in run.get(q, {}).items()} or {"__none__": 0.0} for q in query_ids})
    for ours, theirs in [("nDCG@10", "ndcg_cut_10"), ("Recall@10", "recall_10"), ("Recall@100", "recall_100")]:
        a = per_q[ours].values
        b = np.array([res[q][theirs] for q in per_q.index])
        assert np.allclose(a, b, atol=1e-6), f"{ours} differs from pytrec_eval"
    return "matches pytrec_eval"

def run_from_scores(score_matrix, keep_zero=True):
    # score_matrix: (n_queries, n_docs) dense array -> {qid: {docid: score}} for the top TOP_K
    run = {}
    for i, qid in enumerate(query_ids):
        s = score_matrix[i]
        k = min(TOP_K, len(s))
        top = np.argpartition(-s, k - 1)[:k]
        run[qid] = {doc_ids[j]: float(s[j]) for j in top if keep_zero or s[j] > 0}
    return run

RUNS, PER_QUERY, TIMES = {}, {}, {}

def register(name, run, seconds):
    pq = per_query_metrics(run, qrels)
    RUNS[name], PER_QUERY[name], TIMES[name] = run, pq, seconds
    print(f"{name:<24} nDCG@10 {pq['nDCG@10'].mean():.4f}  R@100 {pq['Recall@100'].mean():.4f}  "
          f"({seconds:.1f}s, {check_with_pytrec(run, pq)})")

## 5. Boolean retrieval

Boolean retrieval returns an unordered set, so ranking metrics do not apply. We evaluate it as a set: how often it returns nothing, how large the set is, and its precision and recall. A query is turned into an AND (every query term must appear) and an OR (any term may appear).

In [ ]:
t0 = time.time()
index = defaultdict(set)
for i, toks in enumerate(doc_tokens):
    for t in set(toks):
        index[t].add(i)

def boolean_set(tokens, mode):
    sets = [index.get(t, set()) for t in set(tokens)]
    if not sets:
        return set()
    return set.intersection(*sets) if mode == "AND" else set.union(*sets)

bool_rows = []
for mode in ["AND", "OR"]:
    sizes, prec, rec = [], [], []
    for qid, toks in zip(query_ids, query_tokens):
        found = {doc_ids[i] for i in boolean_set(toks, mode)}
        rel = set(qrels[qid])
        sizes.append(len(found))
        hit = len(found & rel)
        prec.append(hit / len(found) if found else 0.0)
        rec.append(hit / len(rel))
    sizes = np.array(sizes)
    bool_rows.append({"Query": mode, "Empty result (%)": 100 * np.mean(sizes == 0),
                      "Median docs returned": int(np.median(sizes)),
                      "Precision": np.mean(prec), "Recall": np.mean(rec)})
TIMES["Boolean"] = time.time() - t0
bool_table = pd.DataFrame(bool_rows).set_index("Query").round(3)
bool_table

## 6. Lexical ranking: TF-IDF and BM25

**TF-IDF** weights each term by its frequency in the document and its rarity in the corpus, and ranks by cosine similarity (scikit-learn defaults).

**BM25** [3] is the standard lexical baseline in IR. Unlike TF-IDF it saturates repeated terms and normalises for document length. We use the common defaults k1 = 1.2, b = 0.75, and the Lucene form of IDF. It is added because any new method should be compared against it.

In [ ]:
# TF-IDF
t0 = time.time()
tfidf = TfidfVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None)
D_tfidf = tfidf.fit_transform(doc_texts)
Q_tfidf = tfidf.transform(query_texts)
register("TF-IDF", run_from_scores((Q_tfidf @ D_tfidf.T).toarray(), keep_zero=False), time.time() - t0)

# BM25
t0 = time.time()
K1, B = 1.2, 0.75
cv = CountVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None)
X = cv.fit_transform(doc_texts).tocsr().astype(np.float64)       # docs x terms, raw counts
N = X.shape[0]
dl = np.asarray(X.sum(axis=1)).ravel()
df = np.asarray((X > 0).sum(axis=0)).ravel()
idf = np.log(1 + (N - df + 0.5) / (df + 0.5))
norm = K1 * (1 - B + B * dl / dl.mean())
X = X.tocoo()
w = X.data * (K1 + 1) / (X.data + norm[X.row]) * idf[X.col]
W_bm25 = sparse.csr_matrix((w, (X.row, X.col)), shape=X.shape)
Q_counts = cv.transform(query_texts)
register("BM25", run_from_scores((Q_counts @ W_bm25.T).toarray(), keep_zero=False), time.time() - t0)

## 7. Static word embeddings trained on SciFact: Word2Vec and FastText

Each model is trained on the 5,183 abstracts. A text is represented by the **mean of its word vectors**, and documents are ranked by cosine similarity. FastText also builds vectors for words it has not seen from their character n-grams.

Settings (gensim defaults where possible): 100 dimensions, window 5, min count 2, 10 epochs, one worker thread with a fixed seed so the run is reproducible.

In [ ]:
from gensim.models import Word2Vec, FastText

def mean_vectors(token_lists, vector_of, dim):
    # vector_of(word) returns a vector, or None if the model has no vector for the word
    out = np.zeros((len(token_lists), dim), dtype=np.float32)
    for i, toks in enumerate(token_lists):
        vecs = [v for v in map(vector_of, toks) if v is not None]
        if vecs:
            out[i] = np.mean(vecs, axis=0)
    norms = np.linalg.norm(out, axis=1, keepdims=True)
    return out / np.where(norms == 0, 1, norms)

W2V_ARGS = dict(vector_size=100, window=5, min_count=2, epochs=10, workers=1, seed=SEED)
static_models = [("Word2Vec CBOW", Word2Vec, dict(sg=0)),
                 ("Word2Vec Skip-gram", Word2Vec, dict(sg=1)),
                 ("FastText", FastText, dict(sg=0))]
for name, cls, extra in static_models:
    t0 = time.time()
    model = cls(sentences=doc_tokens, **W2V_ARGS, **extra)
    kv = model.wv
    if name == "FastText":     # builds a vector for any word from its character n-grams
        lookup = lambda t: kv[t]
    else:                      # Word2Vec skips words it did not see in training
        lookup = lambda t: kv[t] if t in kv else None
    Dv = mean_vectors(doc_tokens, lookup, 100)
    Qv = mean_vectors(query_tokens, lookup, 100)
    register(name, run_from_scores(Qv @ Dv.T), time.time() - t0)

## 8. Pretrained models: GloVe, Sentence-BERT and DistilBERT

- **GloVe** `glove-wiki-gigaword-100` [4]: word vectors trained on Wikipedia and news; texts are again the mean of word vectors.
- **Sentence-BERT** `all-MiniLM-L6-v2` [5]: a small transformer fine-tuned to place texts with similar meaning close together.
- **DistilBERT** `distilbert-base-uncased` [6]: a general language model, **not** trained for similarity. Token embeddings are mean-pooled (padding masked out). It shows what a transformer gives without retrieval training.

Transformers read at most 256 subword tokens per document; longer abstracts are truncated.

In [ ]:
if RUN_PRETRAINED:
    import gensim.downloader as gensim_api
    t0 = time.time()
    glove = gensim_api.load("glove-wiki-gigaword-100")
    glove_vec = lambda t: glove[t] if t in glove else None
    register("GloVe", run_from_scores(mean_vectors(query_tokens, glove_vec, 100) @ mean_vectors(doc_tokens, glove_vec, 100).T),
             time.time() - t0)
else:
    print("Skipped (IR_SKIP_PRETRAINED is set).")

In [ ]:
if RUN_PRETRAINED:
    from sentence_transformers import SentenceTransformer
    t0 = time.time()
    sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
    sbert.max_seq_length = 256
    D_sb = sbert.encode(doc_texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    Q_sb = sbert.encode(query_texts, batch_size=64, normalize_embeddings=True)
    register("Sentence-BERT", run_from_scores(Q_sb @ D_sb.T), time.time() - t0)
else:
    print("Skipped (IR_SKIP_PRETRAINED is set).")

In [ ]:
if RUN_PRETRAINED:
    import torch
    from transformers import AutoTokenizer, AutoModel
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    enc = AutoModel.from_pretrained("distilbert-base-uncased").to(DEVICE).eval()

    @torch.no_grad()
    def embed_mean(texts, batch_size=32):
        out = []
        for i in range(0, len(texts), batch_size):
            b = tok(texts[i:i + batch_size], padding=True, truncation=True, max_length=256, return_tensors="pt").to(DEVICE)
            h = enc(**b).last_hidden_state
            m = b["attention_mask"].unsqueeze(-1).float()
            v = (h * m).sum(1) / m.sum(1)
            out.append(torch.nn.functional.normalize(v, dim=1).cpu().numpy())
        return np.vstack(out)

    register("DistilBERT (mean pool)", run_from_scores(embed_mean(query_texts) @ embed_mean(doc_texts).T), time.time() - t0)
else:
    print("Skipped (IR_SKIP_PRETRAINED is set).")

## 9. Results

Mean over the 300 test queries. The 95% confidence interval for nDCG@10 comes from 10,000 bootstrap resamples of queries. The p-value is a paired randomisation test against BM25 on per-query nDCG@10 (10,000 sign flips), with Holm correction across methods.

In [ ]:
rng = np.random.default_rng(SEED)
N_BOOT = 10_000

def bootstrap_ci(x):
    idx = rng.integers(0, len(x), size=(N_BOOT, len(x)))
    means = x[idx].mean(axis=1)
    return np.percentile(means, [2.5, 97.5])

def paired_randomisation(a, b):
    d = a - b
    observed = abs(d.mean())
    signs = rng.choice([-1, 1], size=(N_BOOT, len(d)))
    return (np.sum(np.abs((signs * d).mean(axis=1)) >= observed) + 1) / (N_BOOT + 1)

base = PER_QUERY["BM25"]["nDCG@10"].values
rows = []
for name, pq in PER_QUERY.items():
    x = pq["nDCG@10"].values
    lo, hi = bootstrap_ci(x)
    rows.append({"Method": name, "nDCG@10": x.mean(), "95% CI": f"{lo:.3f}–{hi:.3f}",
                 "Recall@10": pq["Recall@10"].mean(), "Recall@100": pq["Recall@100"].mean(),
                 "MRR@10": pq["MRR@10"].mean(),
                 "p vs BM25": np.nan if name == "BM25" else paired_randomisation(x, base),
                 "Time (s)": TIMES[name]})
results = pd.DataFrame(rows).set_index("Method").sort_values("nDCG@10", ascending=False)

# Holm-Bonferroni across the comparisons with BM25
p = results["p vs BM25"].dropna().sort_values()
m, running = len(p), 0.0
holm = {}
for i, (name, pv) in enumerate(p.items()):
    running = max(running, min(1.0, (m - i) * pv))
    holm[name] = running
results["p (Holm)"] = pd.Series(holm)
results = results.drop(columns="p vs BM25")
results.round(4)

In [ ]:
order = results.index[::-1]
ci = np.array([[float(v) for v in results.loc[n, "95% CI"].split("–")] for n in order])
means = results.loc[order, "nDCG@10"].values
colors = ["#D9622B" if n == "BM25" else "#2B6CB0" for n in order]

fig, ax = plt.subplots(figsize=(8, 0.5 * len(order) + 1.2))
ax.barh(order, means, xerr=[means - ci[:, 0], ci[:, 1] - means], color=colors, capsize=3)
for y, v in enumerate(means):
    ax.text(v + 0.01, y, f"{v:.3f}", va="center", fontsize=9)
ax.set_xlabel("nDCG@10 on SciFact test (300 queries), bars = 95% bootstrap CI")
ax.set_xlim(0, max(1.0, ci.max() + 0.08))
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Retrieval quality by method (BM25 baseline in orange)")
plt.tight_layout()
plt.savefig("scifact_ndcg10.png", dpi=160)
plt.show()

## 10. Where the methods disagree

Per-query comparison of the best neural model with BM25: how many queries each wins, and one example where BM25 misses a relevant abstract that the neural model finds.

In [ ]:
neural = [n for n in ["Sentence-BERT", "DistilBERT (mean pool)", "GloVe"] if n in PER_QUERY]
challenger = max(neural or [n for n in PER_QUERY if n != "BM25"], key=lambda n: PER_QUERY[n]["nDCG@10"].mean())
a, b = PER_QUERY[challenger]["nDCG@10"], PER_QUERY["BM25"]["nDCG@10"]
print(f"{challenger} vs BM25 on nDCG@10: better on {(a > b).sum()}, worse on {(a < b).sum()}, tied on {(a == b).sum()} queries")
print(f"Queries where BM25 finds no relevant abstract in its top 10: {(PER_QUERY['BM25']['Recall@10'] == 0).sum()}")

gap = (a - b).sort_values(ascending=False)
qid = gap.index[0]
qi = query_ids.index(qid)
title = lambda d: (corpus[d][0] or corpus[d][1])[:110]
print(f"\nExample query {qid}: {query_texts[qi]}")
print("Query tokens   :", query_tokens[qi])
print("Relevant       :", [title(d) for d in qrels[qid]])
print(f"BM25 top 1     : {title(ranked_list(RUNS['BM25'][qid])[0]) if RUNS['BM25'][qid] else '(nothing)'}")
print(f"{challenger} top 1 : {title(ranked_list(RUNS[challenger][qid])[0])}")

## 11. Findings

*Written from the outputs above after the full run.*

## 12. Limitations

- **One dataset.** SciFact is small (300 test queries) and scientific; rankings can change on other domains. BEIR covers 18 datasets for this reason.
- **Binary, incomplete judgements.** Unjudged documents count as non-relevant, which can understate methods that retrieve relevant but unjudged abstracts.
- **No tuning.** All methods use defaults. Tuned BM25 or retrieval-trained dense models would score higher; the goal here is a fair comparison of the methods as taught, not a leaderboard.
- **Possible training overlap.** `all-MiniLM-L6-v2` was trained on about one billion text pairs, including scientific-paper data, so some advantage on scientific text may come from related training data.
- **Truncation.** Transformers see the first 256 subword tokens of each abstract.

## References

1. Wadden, D. et al. (2020). *Fact or Fiction: Verifying Scientific Claims.* EMNLP. (SciFact)
2. Thakur, N. et al. (2021). *BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval Models.* NeurIPS Datasets and Benchmarks.
3. Robertson, S. and Zaragoza, H. (2009). *The Probabilistic Relevance Framework: BM25 and Beyond.* Foundations and Trends in IR.
4. Pennington, J., Socher, R. and Manning, C. (2014). *GloVe: Global Vectors for Word Representation.* EMNLP.
5. Reimers, N. and Gurevych, I. (2019). *Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks.* EMNLP.
6. Sanh, V. et al. (2019). *DistilBERT, a distilled version of BERT.* arXiv:1910.01108.
7. Mikolov, T. et al. (2013). *Efficient Estimation of Word Representations in Vector Space.* arXiv:1301.3781. Bojanowski, P. et al. (2017). *Enriching Word Vectors with Subword Information.* TACL. (Word2Vec, FastText)
8. Järvelin, K. and Kekäläinen, J. (2002). *Cumulated Gain-Based Evaluation of IR Techniques.* ACM TOIS. (nDCG)